# Parcel Classifier — Vacant Lot Detection

Train supervised classifiers on NAIP spectral features to identify vacant parcels.

- 16 features: 8 band/index means + 8 stdDevs
- 80/20 stratified split (seed=42)
- Models: Logistic Regression, SVM (RBF), Random Forest
- All use `class_weight="balanced"` for imbalanced classes

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

from vacant_lot.analysis import load_spectral_stats, read_csv_from_gcs
from vacant_lot.config import load_config, _get_shared_root
from vacant_lot.data_utils import load_gdb
from vacant_lot.modeling import (
    build_labels,
    evaluate_classifier,
    get_feature_matrix,
    save_model,
)
from vacant_lot.raster_utils import rasterize_categorical_map

SHARED_ROOT = _get_shared_root()
MODELS_DIR  = SHARED_ROOT / "outputs/modeling/models/nyc_2022"
FIGURES_DIR = SHARED_ROOT / "outputs/modeling/figures/nyc_2022"
DATA_DIR    = SHARED_ROOT / "outputs/modeling/data/nyc_2022"
for d in [MODELS_DIR, FIGURES_DIR, DATA_DIR]:
    d.mkdir(parents=True, exist_ok=True)

In [ ]:
cfg = load_config("nyc_buildings.yaml")

## Load data

In [ ]:
# Load full MapPLUTO parcel dataset with geometry (needed for TIF visualisation)
full_gdf = load_gdb(cfg.get_parcel_path(), layer=cfg.parcel.layer)
full_gdf = full_gdf[["BBL", cfg.parcel.landuse_column, "geometry"]].to_crs(cfg.raster.output_crs)
print(f"Loaded {len(full_gdf):,} total parcels")

In [ ]:
# Load spectral stats from GCS and join with parcel labels
parcel_df = full_gdf[["BBL", cfg.parcel.landuse_column]].copy()
df = load_spectral_stats(
    bucket_name=cfg.gcp.bucket,
    blob_path=f"{cfg.gee.export_prefix}/parcel_spectral_stats.csv",
    parcel_df=parcel_df,
    parcel_id_col="BBL",
    join_cols=["BBL", cfg.parcel.landuse_column],
)
print(f"Spectral stats with labels: {len(df):,} parcels")

In [ ]:
# Build binary labels and feature matrix
y = build_labels(df, cfg)
print(f"Label distribution:\n{y.value_counts()}")

mean_cols = [c for c in df.columns if c.endswith("_mean")]
std_cols  = [c for c in df.columns if c.endswith("_stdDev")]
feature_cols = sorted(mean_cols) + sorted(std_cols)
X, feature_cols = get_feature_matrix(df, feature_cols)
print(f"\nFeatures ({len(feature_cols)}): {feature_cols}")

## Train/test split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42,
)
train_bbls = set(df.loc[X_train.index, "BBL"])
test_bbls  = set(df.loc[X_test.index,  "BBL"])
print(f"Train: {len(X_train):,} ({y_train.mean():.1%} vacant)")
print(f"Test:  {len(X_test):,}  ({y_test.mean():.1%} vacant)")

## Visualise sample (GeoTIFF maps)

Produces two GeoTIFFs for QGIS inspection:
1. **full_sample.tif** — all 25k sampled parcels: not-vacant vs. vacant
2. **train_test_split.tif** — train/test split with vacant highlighted

In [ ]:
sampled_bbls = train_bbls | test_bbls
sample_gdf   = full_gdf[full_gdf["BBL"].isin(sampled_bbls)].copy()
is_vacant    = sample_gdf[cfg.parcel.landuse_column].isin(cfg.parcel.vacant_codes)
sample_gdf["is_vacant"] = is_vacant.astype(int)
print(f"Sampled parcels: {len(sample_gdf):,} ({is_vacant.sum():,} vacant)")

In [ ]:
# TIF 1: full sample — not-vacant vs. vacant
sample_gdf["cat"] = 1
sample_gdf.loc[is_vacant, "cat"] = 2

rasterize_categorical_map(
    sample_gdf, "cat",
    palette={1: (196, 139, 159), 2: (255, 224, 38)},
    output_path=FIGURES_DIR / "full_sample.tif",
)

In [ ]:
# TIF 2: train/test split
in_train = sample_gdf["BBL"].isin(train_bbls)
in_test  = sample_gdf["BBL"].isin(test_bbls)

sample_gdf["cat"] = 0
sample_gdf.loc[in_train & ~is_vacant, "cat"] = 1  # train, not vacant — mauve
sample_gdf.loc[in_train &  is_vacant, "cat"] = 2  # train, vacant — yellow
sample_gdf.loc[in_test  & ~is_vacant, "cat"] = 3  # test,  not vacant — blue
sample_gdf.loc[in_test  &  is_vacant, "cat"] = 4  # test,  vacant — cyan

rasterize_categorical_map(
    sample_gdf[sample_gdf["cat"] > 0], "cat",
    palette={
        1: (196, 139, 159),
        2: (255, 224,  38),
        3: (100, 140, 200),
        4: (  0, 230, 210),
    },
    output_path=FIGURES_DIR / "train_test_split.tif",
)

## Scale features

In [ ]:
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

## Cross-validation

In [ ]:
models = {
    "logreg": LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42),
    "svm":    SVC(kernel="rbf", class_weight="balanced", probability=True, random_state=42),
    "rf":     RandomForestClassifier(
                  n_estimators=200, class_weight="balanced", random_state=42, n_jobs=-1,
              ),
}

cv      = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = ["f1", "average_precision"]

cv_results = []
for name, model in models.items():
    print(f"\n--- {name} ---")
    scores = cross_validate(model, X_train_sc, y_train, cv=cv, scoring=scoring)
    for metric in scoring:
        vals = scores[f"test_{metric}"]
        print(f"  {metric}: {vals.mean():.3f} +/- {vals.std():.3f}")
        cv_results.append({"model": name, "metric": metric,
                           "mean": vals.mean(), "std": vals.std()})

cv_df = pd.DataFrame(cv_results)
cv_df.to_csv(DATA_DIR / "s1_cv_results.csv", index=False)
cv_df

## Train final models and evaluate on test set

In [ ]:
results = {}
for name, model in models.items():
    print(f"\n=== {name} ===")
    model.fit(X_train_sc, y_train)
    results[name] = evaluate_classifier(model, X_test_sc, y_test, model_name=name)

## PR curves

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
for name, res in results.items():
    ax.plot(res["recall"], res["precision"],
            label=f"{name} (AP={res['average_precision']:.3f})")
ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.set_title("Precision-Recall Curves")
ax.legend()
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "s1_pr_curves.png", dpi=150)
plt.show()

## Feature importance (Random Forest)

In [ ]:
rf = models["rf"]
importances = pd.Series(rf.feature_importances_, index=feature_cols).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, 6))
importances.plot.barh(ax=ax)
ax.set_xlabel("Feature Importance")
ax.set_title("Random Forest Feature Importance")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "s1_feature_importance.png", dpi=150)
plt.show()

## Save models

In [ ]:
import joblib

common_meta = {
    "features": feature_cols,
    "n_train": len(X_train),
    "n_test": len(X_test),
    "vacant_fraction_train": float(y_train.mean()),
    "vacant_fraction_test":  float(y_test.mean()),
    "config": "nyc_buildings.yaml",
    "random_state": 42,
}

for name, model in models.items():
    meta = {
        **common_meta,
        "model_type": type(model).__name__,
        "f1": results[name]["f1"],
        "average_precision": results[name]["average_precision"],
    }
    save_model(model, MODELS_DIR / f"s1_{name}_001.joblib", metadata=meta)

joblib.dump(scaler, MODELS_DIR / "s1_scaler_001.joblib")
print(f"Saved scaler → {MODELS_DIR / 's1_scaler_001.joblib'}")